In [ ]:
from pyspark.sql import SparkSession

In [ ]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

In [ ]:
df_green = spark.read.parquet("../data/pq/green/*/*")
df_yellow = spark.read.parquet("../data/pq/yellow/*/*")

In [ ]:
df_yellow.printSchema()

In [ ]:
df_green.createOrReplaceTempView("green_data")
df_yellow.createOrReplaceTempView("yellow_data")

In [ ]:
df_result = spark.sql("""
SELECT 
    -- Reveneue grouping 
    date_trunc('hour', tpep_pickup_datetime) AS revenue_month,
    PULocationID AS revenue_zone,

    SUM(total_amount) AS amount,
    COUNT(1) AS revenue_monthly_total_trips
FROM
    yellow_data
WHERE
    tpep_pickup_datetime >= '2020-01-01' AND tpep_pickup_datetime < '2022-01-01'
GROUP BY
    1, 2
""")

In [ ]:
df_result.show(5)

In [ ]:
df_result.write.parquet("../data/report/revenue/yellow", mode="overwrite")

# Next day (Joins)

In [ ]:
df_yellow_revenue = spark.read.parquet("../data/report/revenue/yellow")
df_green_revenue = spark.read.parquet("../data/report/revenue/green")

In [ ]:
df_green_revenue = df_green_revenue \
    .withColumnRenamed("amount", "green_amount") \
    .withColumnRenamed("revenue_monthly_total_trips", "green_trips")

df_yellow_revenue = df_yellow_revenue \
    .withColumnRenamed("amount", "yellow_amount") \
    .withColumnRenamed("revenue_monthly_total_trips", "yellow_trips")

In [ ]:
df_join = df_green_revenue.join(df_yellow_revenue, on=["revenue_month", "revenue_zone"], how="outer")

In [ ]:
df_join.show(5)

In [ ]:
df_join.write.parquet("../data/report/revenue/total", mode="overwrite")

In [ ]:
df_join = spark.read.parquet("../data/report/revenue/total")

In [ ]:
df_join.show(5)

In [ ]:
df_zones = spark.read.parquet("../data/pq/zones")

In [ ]:
df_zones.show(5)

In [ ]:
df_result = df_join.join(df_zones, df_join.revenue_zone == df_zones.LocationID, how="left")

In [ ]:
df_result.show(5)

In [ ]:
df_result.write.parquet("../data/report/revenue/total_with_zones", mode="overwrite")

In [ ]:
rdd_test = df_result.rdd

In [ ]:
rdd_test.getNumPartitions()